### Data cleaning for scd baseline dataset 

In [658]:
import pandas as pd
import numpy as np

### Phase 1: Diagnose
find the problem before touching anything

#### Step 1: Load & inspect

First rows, shape, dtypes 

In [659]:
df = pd.read_csv("../data/raw/synthetic_scd_baseline_data_MESSY.csv")

df.head(10)

,age,genotype,hbf_percent,ldh_u_l,reticulocyte_pct,indirect_bilirubin_mg_dl,environment,nutrition_status,steady_state_pcv_pct
0,13.2,SS,7.73,779.6,10.20,3.26,urban_poor_access,moderate,14.7
1,2.0,SS,5.79,951.4,15.05,4.8,rural,moderate,14.5
2,9.0,SC,2.46,777.0,12.90,3.98,rural,NaN,21.9
3,20.2,SS,5.22,743.7,10.63,4.74,rural,good,19.3
4,15.8,SS,4.52,504.7,7.29,2.03,rural,moderate,18.5
5,29.7,SS,4.05,870.2,15.11,4.12 mg/dL,rural,moderate,16.6
6,31.6,SC,6.29,702.8,14.09,3.71,rural,moderate,23.9
7,19.4,SS,5.77,773.1,14.86,3.23,urban_poor_access,moderate,20.1
8,11.1,SS,13.29,1123.6,18.98,5.8,urban_good_access,poor,18.3
9,16.9,SS,2.63,712.7,12.49,2.81,urban_poor_access,good,19.2


In [660]:
df.shape

(1530, 9)

In [661]:
df.dtypes

age                         float64
genotype                        str
hbf_percent                 float64
ldh_u_l                         str
reticulocyte_pct            float64
indirect_bilirubin_mg_dl        str
environment                     str
nutrition_status                str
steady_state_pcv_pct        float64
dtype: object

#### Step 2: Check dtype clues

Numeric columns stored as text = contamination hint

In [662]:
# is age genuinely decimal, or just float-typed because of a missing value somewhere?
print(df["age"].isna().sum()) # any missing values?
print((df["age"] % 1 == 0).all())  # True if every value is a whole number

5
False


In [663]:
# find which data parses as numeric
def is_clean_number(x):
    try:
        float(x)
        return True
    except (ValueError, TypeError):
        return False
    
bad_ldh_u_l = df[~df["ldh_u_l"].apply(is_clean_number)]
print(bad_ldh_u_l[["ldh_u_l"]].head(3))
print(f"\n{len(bad_ldh_u_l)} contaminated rows out of {len(df)}")

        ldh_u_l
68    833.8 U/L
97   1132.8 U/L
101  1039.7 U/L

76 contaminated rows out of 1530


In [664]:
bad_indirect_bilirubin_mg_dl = df[~df["indirect_bilirubin_mg_dl"].apply(is_clean_number)]
print(bad_indirect_bilirubin_mg_dl[["indirect_bilirubin_mg_dl"]].head(3))
print(f"\n{len(bad_indirect_bilirubin_mg_dl)} contaminated rows out of {len(df)}")

   indirect_bilirubin_mg_dl
5                4.12 mg/dL
33               2.72 mg/dL
44                4.9 mg/dL

75 contaminated rows out of 1530


### Step 4: Profile every column

Missing %, unique count, min/max, sample values

Check for: 
1. Missing values
2. Categorical consistency (are labels spelled consistently?)
3. Numeric plausibility (are values within realistic ranges?)
4. Type contamination (are "numbers" secretly stored as text?)

In [665]:
# Mini Profiling
for col in df.columns:
    print(f"\n--- {col} ---")
    print(f"dtype: {df[col].dtype}")
    print(f"missing: {df[col].isna().sum()} ({df[col].isna().sum() / len(df) * 100:.1f}%)")
    print(f"unique values: {df[col].nunique()}")
    if df[col].dtype in ["int64", "float64"]:
        print(f"min: {df[col].min()}, max: {df[col].max()}")
    else:
        print(f"sample values: {df[col].dropna().unique()[:8]}")


--- age ---
dtype: float64
missing: 5 (0.3%)
unique values: 400
min: -3.0, max: 58.2

--- genotype ---
dtype: str
missing: 5 (0.3%)
unique values: 16
sample values: <StringArray>
['SS', 'SC', ' SS ', 'Hb SS', 'Hb-SS', 'SBeta-thal', 'ss', 'HBSS']
Length: 8, dtype: str

--- hbf_percent ---
dtype: float64
missing: 65 (4.2%)
unique values: 851
min: 0.5, max: 30.0

--- ldh_u_l ---
dtype: str
missing: 66 (4.3%)
unique values: 1327
sample values: <StringArray>
['779.6', '951.4', '777.0', '743.7', '504.7', '870.2', '702.8', '773.1']
Length: 8, dtype: str

--- reticulocyte_pct ---
dtype: float64
missing: 66 (4.3%)
unique values: 931
min: -5.0, max: 24.08

--- indirect_bilirubin_mg_dl ---
dtype: str
missing: 66 (4.3%)
unique values: 594
sample values: <StringArray>
['3.26', '4.8', '3.98', '4.74', '2.03', '4.12 mg/dL', '3.71', '3.23']
Length: 8, dtype: str

--- environment ---
dtype: str
missing: 5 (0.3%)
unique values: 11
sample values: <StringArray>
[ 'urban_poor_access',              'rural',

### Step 4: Check for duplicates
Column profiling alone never catches a duplicate row

In [666]:
print(df.duplicated().sum())

29


### Phase 2: Fix steps
Correct what the diagnosis found

#### Step 5: Copy before editing

Keep the raw file untouched

In [667]:
df_clean = df.copy()
df_clean

,age,genotype,hbf_percent,ldh_u_l,reticulocyte_pct,indirect_bilirubin_mg_dl,environment,nutrition_status,steady_state_pcv_pct
0,13.2,SS,7.73,779.6,10.20,3.26,urban_poor_access,moderate,14.7
1,2.0,SS,5.79,951.4,15.05,4.8,rural,moderate,14.5
2,9.0,SC,2.46,777.0,12.90,3.98,rural,NaN,21.9
3,20.2,SS,5.22,743.7,10.63,4.74,rural,good,19.3
4,15.8,SS,4.52,504.7,7.29,2.03,rural,moderate,18.5
...,...,...,...,...,...,...,...,...,...
1525,17.4,SS,2.40,NaN,15.38,5.94,urban_poor_access,good,13.1
1526,35.6,SS,3.74,995.2,16.77,5.44,urban_good_access,poor,13.0
1527,29.8,SS,3.67,678.8,12.57,2.99,urban_good_access,moderate,20.0
1528,10.6,SS,3.87,978.1,12.71,5.52,rural,poor,12.0


In [668]:
df_clean.shape

(1530, 9)

#### Step 6: Drop empty rows

rows with nothing recorded at all

In [669]:
df_clean = df_clean.dropna(how="all") 
df_clean.shape

(1525, 9)

#### Step 7: Drop duplicate rows

Exact repeat, keep the first

In [670]:
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
df_clean.shape

(1500, 9)

In [671]:
df_clean["ldh_u_l"].isna().sum()

np.int64(60)

#### Step 8: Fix type contamination

Strip stray text/units, restore real numbers

In [672]:
print("Before fixing type")
print("ldh_u_l: ", df_clean["ldh_u_l"].dtypes)
print("indirect_bilirubin_mg_dl: ", df_clean["indirect_bilirubin_mg_dl"].dtypes)

for col in ["ldh_u_l", "indirect_bilirubin_mg_dl"]:
    df_clean[col] = df_clean[col].astype(str).str.extract(r"(-?\d+\.?\d*)")[0]
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")
    
print("After fixing type")
print("ldh_u_l: ", df_clean["ldh_u_l"].dtypes)
print("indirect_bilirubin_mg_dl: ", df_clean["indirect_bilirubin_mg_dl"].dtypes)


Before fixing type
ldh_u_l:  str
indirect_bilirubin_mg_dl:  str
After fixing type
ldh_u_l:  float64
indirect_bilirubin_mg_dl:  float64


In [673]:
df_clean["ldh_u_l"].isna().sum()

np.int64(60)

In [674]:
df_clean["indirect_bilirubin_mg_dl"].isna().sum()

np.int64(60)

In [675]:
pd.DataFrame([
    6,
    np.nan,
    np.nan,
    4
]).sum()

0    10.0
dtype: float64

#### Step 9: Normalized categories

Collapse spelling variants into clean labels

In [676]:
# Normalize genotype

print(df_clean["genotype"].unique())
print(df_clean["genotype"].dropna().unique())

<StringArray>
[         'SS',          'SC',        ' SS ',       'Hb SS',       'Hb-SS',
  'SBeta-thal',          'ss',        'HBSS',          'sc', 'S-beta thal',
        'HBSC',       'Hb-SC',  'sbeta-thal',        ' SC ', 'S/beta-thal',
  'Sbeta thal']
Length: 16, dtype: str
<StringArray>
[         'SS',          'SC',        ' SS ',       'Hb SS',       'Hb-SS',
  'SBeta-thal',          'ss',        'HBSS',          'sc', 'S-beta thal',
        'HBSC',       'Hb-SC',  'sbeta-thal',        ' SC ', 'S/beta-thal',
  'Sbeta thal']
Length: 16, dtype: str


In [677]:
genotype_map = {}

for val in df_clean["genotype"].dropna().unique():
    v = str(val).strip().lower().replace("hb ", "").replace("hb-", "").replace("hb", "")
    
    if v == "ss":
        genotype_map[val] = "SS"
    elif v == "sc":
        genotype_map[val] = "SC"
    elif "beta" in v:
        genotype_map[val] = "SBeta-thal"

genotype_map

{'SS': 'SS',
 'SC': 'SC',
 ' SS ': 'SS',
 'Hb SS': 'SS',
 'Hb-SS': 'SS',
 'SBeta-thal': 'SBeta-thal',
 'ss': 'SS',
 'HBSS': 'SS',
 'sc': 'SC',
 'S-beta thal': 'SBeta-thal',
 'HBSC': 'SC',
 'Hb-SC': 'SC',
 'sbeta-thal': 'SBeta-thal',
 ' SC ': 'SC',
 'S/beta-thal': 'SBeta-thal',
 'Sbeta thal': 'SBeta-thal'}

In [678]:
df_clean["genotype"] = df_clean["genotype"].map(genotype_map)
print(df_clean["genotype"].unique())

<StringArray>
['SS', 'SC', 'SBeta-thal']
Length: 3, dtype: str


In [679]:
# Normalize environment

print(df_clean["environment"].unique())
print(df_clean["environment"].dropna().unique())

<StringArray>
[ 'urban_poor_access',              'rural',  'urban_good_access',
 ' urban_good_access',              'Rural',            ' rural ',
              'RURAL',  'Urban Poor Access',  'URBAN_GOOD_ACCESS',
  'Urban Good Access', ' urban_poor_access']
Length: 11, dtype: str
<StringArray>
[ 'urban_poor_access',              'rural',  'urban_good_access',
 ' urban_good_access',              'Rural',            ' rural ',
              'RURAL',  'Urban Poor Access',  'URBAN_GOOD_ACCESS',
  'Urban Good Access', ' urban_poor_access']
Length: 11, dtype: str


In [680]:
environment_map = {}

for val in df_clean["environment"].dropna().unique():
    v = str(val).strip().lower().replace(" ", "_")
    environment_map[val] = v
    
environment_map  


{'urban_poor_access': 'urban_poor_access',
 'rural': 'rural',
 'urban_good_access': 'urban_good_access',
 ' urban_good_access': 'urban_good_access',
 'Rural': 'rural',
 ' rural ': 'rural',
 'RURAL': 'rural',
 'Urban Poor Access': 'urban_poor_access',
 'URBAN_GOOD_ACCESS': 'urban_good_access',
 'Urban Good Access': 'urban_good_access',
 ' urban_poor_access': 'urban_poor_access'}

In [681]:
df_clean["environment"] = df_clean["environment"].map(environment_map)

In [682]:
print(df_clean["environment"].unique())

<StringArray>
['urban_poor_access', 'rural', 'urban_good_access']
Length: 3, dtype: str


In [683]:
# Normalize nutrition_status

print(df_clean["nutrition_status"].unique())
print(df_clean["nutrition_status"].dropna().unique())


<StringArray>
[ 'moderate',         nan,      'good',      'poor', ' moderate',      'GOOD',
      'Good',      'Poor',     'POOR ',  'Moderate']
Length: 10, dtype: str
<StringArray>
[ 'moderate',      'good',      'poor', ' moderate',      'GOOD',      'Good',
      'Poor',     'POOR ',  'Moderate']
Length: 9, dtype: str


In [684]:
nutrition_status_map = {}

for val in df_clean["nutrition_status"].dropna().unique():
    v = str(val).strip().lower()
    nutrition_status_map[val] = v
    
nutrition_status_map

{'moderate': 'moderate',
 'good': 'good',
 'poor': 'poor',
 ' moderate': 'moderate',
 'GOOD': 'good',
 'Good': 'good',
 'Poor': 'poor',
 'POOR ': 'poor',
 'Moderate': 'moderate'}

In [685]:
df_clean["nutrition_status"] = df_clean["nutrition_status"].map(nutrition_status_map)

print(df_clean["nutrition_status"].unique())
print(df_clean["nutrition_status"].dropna().unique())

<StringArray>
['moderate', nan, 'good', 'poor']
Length: 4, dtype: str
<StringArray>
['moderate', 'good', 'poor']
Length: 3, dtype: str


#### Step 10: Null out impossible values

Outside any physically plausible range

In [686]:
# Covered by RANGE - Just alternative
df_clean[df_clean["age"] < 0]

,age,genotype,hbf_percent,ldh_u_l,reticulocyte_pct,indirect_bilirubin_mg_dl,environment,nutrition_status,steady_state_pcv_pct
528,-3.0,SBeta-thal,4.84,868.1,11.20,3.68,urban_poor_access,good,25.3
667,-3.0,SS,2.82,688.7,10.58,4.31,urban_good_access,moderate,17.4
1110,-3.0,SS,3.99,1019.9,17.56,5.63,rural,moderate,18.5


In [687]:
df_clean.loc[df_clean["age"] < 0, "age"] = np.nan

In [688]:
# Plausible ranges (inclusive). Values outside become NaN.
RANGES = {
    "age": (0, 100),
    "steady_state_pcv_pct": (10, 50),
    "reticulocyte_pct": (0, 40),
    "ldh_u_l": (50, 3000),
    "indirect_bilirubin_mg_dl": (0, 20),
    "hbf_percent": (0, 100),
}

In [689]:
for col, (low, high) in RANGES.items():
    bad = ~df_clean[col].between(low, high) & df_clean[col].notna()
    print(f"{col}: {bad.sum()} out of range")
    df_clean.loc[bad, col] = np.nan

age: 0 out of range
steady_state_pcv_pct: 5 out of range
reticulocyte_pct: 3 out of range
ldh_u_l: 4 out of range
indirect_bilirubin_mg_dl: 0 out of range
hbf_percent: 0 out of range


#### Step 11: Drop rows missing the label

Can't input a target you're predicting

In [690]:
df_clean["steady_state_pcv_pct"].isna().sum()

np.int64(5)

In [691]:
df_clean = df_clean.dropna(subset=["steady_state_pcv_pct"]).reset_index(drop=True)

In [692]:
df_clean["steady_state_pcv_pct"].isna().sum()

np.int64(0)

#### Step 12: Input what's left

Grouped by a relevant category, not one average

In [693]:
pd.DataFrame(df_clean.groupby("genotype"))

,0,1
0,SBeta-thal,age genotype hbf_percent ldh_u_l ...
1,SC,age genotype hbf_percent ldh_u_l ret...
2,SS,age genotype hbf_percent ldh_u_l ret...


In [694]:
pd.DataFrame(df_clean.groupby("genotype")["age"])

,0,1
0,SBeta-thal,39 9.3 48 21.9 71 34.1 72 ...
1,SC,2 9.0 6 31.6 10 23.4 14 ...
2,SS,0 13.2 1 2.0 3 20.2 4 ...


In [695]:
df_clean.groupby("genotype")["age"].transform(lambda s: s.fillna(s.median()))

0       13.2
1        2.0
2        9.0
3       20.2
4       15.8
        ... 
1490    17.4
1491    35.6
1492    29.8
1493    10.6
1494    39.3
Name: age, Length: 1495, dtype: float64

In [696]:
numeric_cols = ["age", "hbf_percent", "ldh_u_l", "reticulocyte_pct", "indirect_bilirubin_mg_dl"]

for col in numeric_cols:
    df_clean[col] = df_clean.groupby("genotype")[col].transform(lambda s: s.fillna(s.median()))

In [697]:
print(df_clean["genotype"].mode()[0])
print(df_clean["environment"].mode()[0])
print(df_clean["nutrition_status"].mode()[0])

SS
urban_good_access
good


In [698]:
for col in ["genotype", "environment", "nutrition_status"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

### Phase 3: Verify 

Confirm the fixes actually worked

#### Step 13: Final checks

Confirm 0 missing, 0 duplicates, clean categories

In [699]:
print(f"Final shape: {df_clean.shape}")
print(f"Remaining missing values: {df_clean.isna().sum().sum()}")
print(f"Remaining duplicates: {df_clean.duplicated().sum()}")

Final shape: (1495, 9)
Remaining missing values: 0
Remaining duplicates: 0


In [700]:
df_clean.isna().sum()

age                         0
genotype                    0
hbf_percent                 0
ldh_u_l                     0
reticulocyte_pct            0
indirect_bilirubin_mg_dl    0
environment                 0
nutrition_status            0
steady_state_pcv_pct        0
dtype: int64

In [701]:
df_clean.loc[df_clean["indirect_bilirubin_mg_dl"].isna()]

,age,genotype,hbf_percent,ldh_u_l,reticulocyte_pct,indirect_bilirubin_mg_dl,environment,nutrition_status,steady_state_pcv_pct


### Step 14: Saved cleaned file
Seperate from the raw original

In [702]:
# save the cleaned dataset

df_clean.to_csv("../data/processed/synthetic_scd_baseline_data_CLEANED.csv", index=False)